# Chat con el bot de recetas fitosanitarias

Conversá con el bot desde acá, con el mismo orquestador que usa WhatsApp: Gemini elige las tools, la base de Postgres tiene el registro de SENASA y la normativa, y las respuestas salen con el mismo formato que en WhatsApp.

**Antes de empezar**
- La base levantada: `docker compose up -d db`.
- En `.env`, al menos una `GEMINI_API_KEY_*` (con varias, el bot rota de key si una se queda sin cuota).
- Correr la celda de preparación (tarda unos 30 s la primera vez: carga el modelo de embeddings).

**Cómo usarlo**
- Con el chat de abajo: escribí y apretá *Enter* o *Enviar*; para una receta, elegí la foto con *Foto* y enviá (con o sin texto). Cuando el bot ofrece opciones (`[Confirmar] [Corregir]`, `[Agendar] [No, gracias]`), aparecen como botones.
- O sin widgets: `enviar("tu mensaje")`, `enviar(foto="ruta/a/la/foto.jpg")` y `nueva()` para empezar otra conversación.
- Cada conversación queda en el historial del bot (como un chat de WhatsApp): para cambiar de tema conviene *Nueva conversación*.

In [ ]:
import os
import sys
from pathlib import Path

# Jupyter arranca en notebooks/: nos paramos en la raíz del proyecto para que se lea .env.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd() / "src"))
os.environ["USE_FIXTURES"] = "false"  # el bot de verdad: Gemini y la base

from IPython.display import Markdown, display

from fitosanitarios.canales.notebook import Conversacion, a_markdown
from fitosanitarios.servicios.recursos import precargar_modelo_embeddings

precargar_modelo_embeddings()
bot = Conversacion()


def mostrar_turno(texto: str = "", foto=None, nombre_foto: str | None = None) -> list[str]:
    adjunto = f" 📷 `{nombre_foto or foto}`" if foto is not None else ""
    display(Markdown(f"**🧑‍🌾 Vos:** {texto}{adjunto}"))
    mensajes = bot.enviar(texto, foto)
    for m in mensajes:
        display(Markdown("**🤖 Bot:**  \n" + a_markdown(m)))
    return mensajes


def enviar(texto: str = "", foto=None) -> None:
    """Manda un mensaje al bot y muestra la respuesta. `foto`: ruta o bytes de una imagen."""
    mostrar_turno(texto, foto)


def nueva() -> None:
    """Empieza otra conversación: el bot no recuerda la anterior."""
    print(f"Conversación nueva: {bot.nueva()}")


print(f"Listo. Conversación: {bot.thread_id}")

## Chat

Escribí abajo. Los botones con las opciones del bot mandan esa opción como respuesta.

In [ ]:
import re
import warnings

import ipywidgets as w

historial = w.Output(layout=w.Layout(
    border="1px solid #ccc", padding="8px", max_height="520px", overflow="auto"
))
entrada = w.Text(placeholder="Escribí tu mensaje y apretá Enter", layout=w.Layout(width="60%"))
selector_foto = w.FileUpload(accept="image/*", multiple=False, description="Foto")
boton_enviar = w.Button(description="Enviar", button_style="primary")
boton_nueva = w.Button(description="Nueva conversación")
opciones = w.HBox()
estado = w.Label()


def _botones_de(mensajes: list[str]) -> list[str]:
    """Las opciones que ofrece el último mensaje: `[Confirmar] [Corregir]`."""
    return re.findall(r"\[([^\]\n]+)\]", mensajes[-1]) if mensajes else []


def _mandar(texto: str, foto=None, nombre_foto=None) -> None:
    opciones.children = ()
    estado.value = "El bot está pensando..."
    try:
        with historial:
            mensajes = mostrar_turno(texto, foto, nombre_foto)
    finally:
        estado.value = ""
    botones = []
    for opcion in _botones_de(mensajes):
        boton = w.Button(description=opcion)
        boton.on_click(lambda _, o=opcion: _mandar(o))
        botones.append(boton)
    opciones.children = tuple(botones)


def _al_enviar(_=None) -> None:
    texto = entrada.value.strip()
    foto = nombre = None
    if selector_foto.value:
        archivo = selector_foto.value[0]
        foto, nombre = archivo["content"].tobytes(), archivo["name"]
    if not texto and foto is None:
        return
    entrada.value = ""
    selector_foto.value = ()
    _mandar(texto, foto, nombre)


def _al_empezar_de_nuevo(_=None) -> None:
    historial.clear_output()
    opciones.children = ()
    with historial:
        nueva()


boton_enviar.on_click(_al_enviar)
boton_nueva.on_click(_al_empezar_de_nuevo)
with warnings.catch_warnings():
    # `on_submit` está deprecado, pero es el único que se dispara solo con Enter (con
    # `observe` también manda el mensaje al salir del campo, p. ej. al elegir la foto).
    warnings.simplefilter("ignore")
    entrada.on_submit(_al_enviar)

display(w.VBox([
    historial, opciones, w.HBox([entrada, selector_foto, boton_enviar]), boton_nueva, estado,
]))

## Sin widgets: ejemplos

Cada celda manda un mensaje en la conversación actual. Algunos para probar:

In [ ]:
enviar("Hola, ¿qué podés hacer?")

In [ ]:
enviar("¿Qué fungicidas para trigo puedo aplicar con avión a 1500 metros de El Trébol?")

In [ ]:
nueva()
enviar(foto="data/recetas_ejemplo/02_apta_aerea_banda_ii.jpg")

In [ ]:
enviar("en El Trébol")

## Al terminar

Cierra la conexión del historial con la base.

In [ ]:
bot.cerrar()